# Clinical search comparison

Run one clinical question through **three literature-search lanes** and compare their answers side by side.

| Lane | How it answers |
|------|----------------|
| **PubMed (LLM-curated)** | Claude builds a curated PubMed query → NCBI E-utilities → Claude synthesizes a grounded answer citing `[PMID:…]`. |
| **Consensus** | Consensus `/v1/quick_search` ranked papers → Claude synthesizes an answer from them. (Needs `CONSENSUS_API_KEY`.) |
| **OpenEvidence** | Configurable adapter for OpenEvidence's gated enterprise API. (Needs enterprise access + BAA.) |

Configure keys in `.env` (see `.env.example`). A lane that isn't configured shows a `not_configured` row — it never fabricates an answer.

**Run `Cell → Run All`, then type a question and click _Run comparison_.**

In [1]:
# Setup: load .env, import the engine, show which lanes are ready.
import importlib
from dotenv import load_dotenv

load_dotenv()  # reads .env in this folder

import search_lanes
importlib.reload(search_lanes)  # pick up edits without restarting the kernel
from search_lanes import compare, lane_status, results_to_html

from IPython.display import HTML, display

status = lane_status()
ready_icon, gated_icon = "✅", "\U0001f512"
rows = "".join(
    f"<div style='margin:2px 0'>{ready_icon if v == 'ready' else gated_icon} "
    f"<b>{k}</b> &mdash; {v}</div>"
    for k, v in status.items()
)
display(HTML("<h4 style='margin-bottom:6px'>Lane status</h4>" + rows))

In [2]:
# Interactive UI: enter a question, run the three lanes, render the comparison table.
import ipywidgets as widgets
from IPython.display import clear_output

question = widgets.Textarea(
    value=(
        "In adults with type 2 diabetes, does SGLT2 inhibitor therapy "
        "reduce cardiovascular mortality?"
    ),
    placeholder="Enter a clinical question\u2026",
    description="Question:",
    layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "initial"},
)
run_btn = widgets.Button(description="Run comparison", button_style="primary", icon="search")
out = widgets.Output()


def on_run(_):
    with out:
        clear_output()
        q = question.value.strip()
        if not q:
            print("Please enter a question.")
            return
        print("Running 3 lanes\u2026 (this calls the LLM + live APIs)")
        results = compare(q)
        clear_output()
        display(HTML(results_to_html(results)))


run_btn.on_click(on_run)
display(widgets.VBox([question, run_btn, out]))

## Enabling the gated lanes

- **Consensus** — apply at <https://consensus.app/home/api/>, then set `CONSENSUS_API_KEY` in `.env`.
- **OpenEvidence** — enterprise + signed BAA only (<sales@openevidence.com>). Its API schema isn't public, so set `OPENEVIDENCE_API_BASE`, `OPENEVIDENCE_API_KEY`, `OPENEVIDENCE_ORG_ID`, and (if needed) `OPENEVIDENCE_SEARCH_PATH` / `OPENEVIDENCE_ANSWER_FIELD` / `OPENEVIDENCE_SOURCES_FIELD` to match the real spec.

After editing `.env`, re-run the **Setup** cell (it reloads the engine and re-reads the environment).